# DMEyF 2026 - Primera Competencia: modelo final (rank_signo, 20 semillas)

Predicción de **BAJA+2** del Paquete Premium. Este notebook reproduce, desde el CSV crudo, el archivo que se entregó:
**`ensamble_K13000.csv`** (13.000 clientes de 202108), con una ganancia pública de **105,02**.

| Etapa | Qué hace |
|---|---|
| 0 | Carga el CSV crudo y calcula `clase_ternaria` |
| 1.1 | Data Quality: `ccajas_depositos` rota en 202105 (todos ceros) -> `NA` solo en ese mes |
| 1.2 | Data Drifting: se eliminan 12 variables (`cprestamos_personales`, `mprestamos_personales` y las 10 con mayor KS entre 202103-05 y 202108) y se aplica **ranking mensual** a las variables monetarias `m*` (de -1 a 1 con el cero fijo si tienen valores negativos; de 0 a 1 si no). El ranking se hace **antes** de calcular lags y deltas |
| 1.4 | Feature Engineering histórico: lags y delta-lags de orden 1 y 2 |
| 1.7 | Bayesian Optimization (Optuna/TPE, AUC, sin undersampling): ver la sección correspondiente |
| 2.1 | Final training: 20 semillas con los mismos hiperparámetros, en 202103-202106, sin undersampling |
| 2.2 | Scoring de 202108 y ensamble promediando las probabilidades por `numero_de_cliente` |
| 2.3 | Cortes de envíos K (12.500 a 14.500) para probar contra el pseudo Kaggle |

Ganancia: acierto (BAJA+2 enviado) = **+1.072.500**, envío inútil = **-27.500**.
La clase de entrenamiento es binaria, POS = {BAJA+1, BAJA+2} y NEG = CONTINUA; la ganancia solo cuenta los BAJA+2.

## Configuración

In [1]:
import json, time, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| lightgbm", lgb.__version__)

_hay = lambda d: (d / "competencia_01_crudo.csv").exists() or (d / "competencia_01.parquet").exists()
DATA_DIR = Path("datos") if _hay(Path("datos")) else Path("..") / "datos"   # carpeta con competencia_01_crudo.csv
RAW = DATA_DIR / "competencia_01_crudo.csv"
CACHE = DATA_DIR / "competencia_01.parquet"
OUT_DIR = Path("salida")                            # resultados de este notebook
OUT_DIR.mkdir(parents=True, exist_ok=True)

MESES_FINAL = [202103, 202104, 202105, 202106]
MES_APLICAR = 202108
MESES_USO = sorted(set(MESES_FINAL + [MES_APLICAR]))
GANANCIA_ACIERTO, COSTO_ENVIO = 1_072_500, -27_500

# 12 variables eliminadas: prestamos_personales (la catedra las marca como data drifting) y las 10 con mayor KS
VARIABLES_ELIMINADAS = ["cprestamos_personales", "mprestamos_personales",
                        "Master_fultimo_cierre", "Visa_fultimo_cierre", "Master_Finiciomora", "Visa_Finiciomora",
                        "mcomisiones_mantenimiento", "Visa_mpagado", "mrentabilidad", "Master_Fvencimiento",
                        "mpasivos_margen", "Visa_Fvencimiento"]
EXCLUIR = {"numero_de_cliente", "foto_mes", "periodo0", "clase_ternaria"}

# 20 semillas (las 5 propias, primos entre 100003 y 999983, y 15 primos al azar con la semilla 285937)
SEMILLAS = [285937, 474619, 502487, 600673, 666637, 326219, 213727, 962341, 884959, 305759,
            274583, 556123, 182129, 752489, 852199, 362927, 130307, 212227, 301331, 915727]
KS = list(range(12500, 14501, 250))                 # cortes de envios a probar
K_ENTREGA = 13000                                   # corte entregado

# Hiperparametros: los fijos de la catedra y los 3 optimizados por la Bayesian Optimization (trial 27, ver seccion 1.7)
PARAMS = dict(objective="binary", metric="auc", boosting="gbdt", verbosity=-1,
              max_bin=31, min_data_in_leaf=0, feature_fraction=0.50, feature_fraction_bynode=0.20,
              learning_rate=0.005, feature_pre_filter=False, force_row_wise=True,
              num_leaves=171, min_sum_hessian_in_leaf=7.28721427391867)
NUM_ITERATIONS = 1050
DS_PARAMS = dict(max_bin=31, min_data_in_leaf=0, feature_pre_filter=False)

python 3.12.10 | numpy 2.5.3 | pandas 3.0.6 | lightgbm 4.7.0


## 0. Dataset y clase_ternaria
El CSV crudo no trae la clase. Se calcula mirando hacia el futuro, igual que el script en R de la cátedra:
`BAJA+1` si el cliente no está en el mes siguiente, `BAJA+2` si está el mes siguiente pero no el subsiguiente, `CONTINUA` el resto.
Los últimos dos meses del dataset quedan sin clase completa (por eso solo se entrena hasta 202106).

In [2]:
if CACHE.exists():
    df = pd.read_parquet(CACHE)
else:
    df = pd.read_csv(RAW, engine="pyarrow")
    df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
    df["periodo0"] = (df.foto_mes // 100) * 12 + df.foto_mes % 100
    g = df.groupby("numero_de_cliente")["periodo0"]
    p1, p2 = g.shift(-1), g.shift(-2)
    ultimo = df.periodo0.max(); anteultimo = ultimo - 1
    clase = np.full(len(df), None, dtype=object)
    clase[(df.periodo0 < anteultimo).to_numpy()] = "CONTINUA"
    m1 = (df.periodo0 < ultimo) & (p1.isna() | (df.periodo0 + 1 < p1))
    clase[m1.to_numpy()] = "BAJA+1"
    m2 = (df.periodo0 < anteultimo) & (df.periodo0 + 1 == p1) & (p2.isna() | (df.periodo0 + 2 < p2))
    clase[m2.to_numpy()] = "BAJA+2"
    df["clase_ternaria"] = clase
    for c in df.columns:                              # float32 para ahorrar memoria
        if c not in ("numero_de_cliente", "foto_mes", "periodo0", "clase_ternaria") and pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype("float32")
    df.to_parquet(CACHE)
print(df.shape)
df.groupby(["foto_mes", "clase_ternaria"], dropna=False).size().unstack(fill_value=0)

(983061, 156)


clase_ternaria,BAJA+1,BAJA+2,CONTINUA,NaN
foto_mes,,,,
202103,1019,960,160921,0
202104,964,1139,161181,0
202105,1143,870,161755,0
202106,874,1098,162142,0
202107,1103,0,0,163245
202108,0,0,0,164647


## 1.1 Data Quality
`ccajas_depositos` está rota en 202105 (todos ceros) -> se asigna `NA` solo para ese mes.

In [3]:
print("antes: valores no cero en 202105 =", int((df.loc[df.foto_mes == 202105, "ccajas_depositos"] != 0).sum()))
df.loc[df.foto_mes == 202105, "ccajas_depositos"] = np.nan
print("despues: valores no nulos en 202105 =", int(df.loc[df.foto_mes == 202105, "ccajas_depositos"].notna().sum()))

antes: valores no cero en 202105 = 0
despues: valores no nulos en 202105 = 0


## 1.2 Data Drifting
Se eliminan 12 variables (listadas en la configuración) y se aplica **ranking mensual** a las variables monetarias `m*`:
- las que tienen valores negativos pasan a un rango de **-1 a 1 con el cero fijo** (los positivos se rankean entre sí de 0 a 1, los negativos de -1 a 0);
- las que no tienen negativos pasan a un rango de **0 a 1**.

El ranking se calcula dentro de cada `foto_mes` y **antes** de calcular los lags y deltas.

In [4]:
df = df.drop(columns=[c for c in VARIABLES_ELIMINADAS if c in df.columns])
MONETARIAS = [c for c in df.columns if c.startswith("m")]
CON_NEGATIVOS = [c for c in MONETARIAS if (df[c] < 0).any()]
SIN_NEGATIVOS = [c for c in MONETARIAS if c not in CON_NEGATIVOS]
print("monetarias rankeadas:", len(MONETARIAS), "| con negativos (-1 a 1):", len(CON_NEGATIVOS), "| sin negativos (0 a 1):", len(SIN_NEGATIVOS))

df[SIN_NEGATIVOS] = df.groupby("foto_mes")[SIN_NEGATIVOS].rank(pct=True).astype("float32")
mes = df["foto_mes"]
for c in CON_NEGATIVOS:
    x = df[c]
    pos = x.where(x > 0).groupby(mes).rank(pct=True)          # positivos: 0 a 1
    neg = (-x).where(x < 0).groupby(mes).rank(pct=True)       # negativos: -1 a 0
    out = np.where(x > 0, pos, np.where(x < 0, -neg, 0.0))    # el cero queda fijo en 0
    out[x.isna().to_numpy()] = np.nan                         # los NA siguen siendo NA
    df[c] = out.astype("float32")

monetarias rankeadas: 41 | con negativos (-1 a 1): 10 | sin negativos (0 a 1): 31


## 1.4 Feature Engineering histórico
Lags y delta-lags (valor actual menos valor del lag) de orden 1 y 2 para todas las variables numéricas (calculados sobre los rangos de la sección anterior).
Se calculan alineando por (cliente, período) y no por posición, para no cruzar meses si a un cliente le falta alguno.

In [5]:
cols_base = [c for c in df.columns if c not in EXCLUIR]

def agregar_lags(df, meses_uso, cols, ks=(1, 2)):
    clave = ["numero_de_cliente", "periodo0"]
    uso = df[df.foto_mes.isin(meses_uso)].copy()
    base = df[clave + cols]
    nuevas = []
    for k in ks:
        r = base.copy()
        r["periodo0"] = r["periodo0"] + k                 # el valor de hace k meses cae en el periodo actual
        r.columns = clave + [f"{c}_lag{k}" for c in cols]
        lag = uso[clave].merge(r, on=clave, how="left").drop(columns=clave)
        lag.index = uso.index
        delta = pd.DataFrame(uso[cols].to_numpy() - lag.to_numpy(),
                             columns=[f"{c}_dlag{k}" for c in cols], index=uso.index)
        nuevas += [lag, delta]
    return pd.concat([uso] + nuevas, axis=1)

t0 = time.time()
ds = agregar_lags(df, MESES_USO, cols_base)
del df
FEATURES = [c for c in ds.columns if c not in EXCLUIR]           # sin numero_de_cliente ni foto_mes
ds["y_pos"] = ds.clase_ternaria.isin(["BAJA+1", "BAJA+2"]).astype("int8")   # clase de entrenamiento
ds["y_gan"] = (ds.clase_ternaria == "BAJA+2").astype("int8")                # clase que da la ganancia
print(f"{ds.shape[0]:,} filas, {len(FEATURES)} features, {time.time() - t0:.0f}s")

818,713 filas, 700 features, 8s


## 1.7 Bayesian Optimization
Los 3 hiperparámetros optimizados (`num_iterations`, `num_leaves`, `min_sum_hessian_in_leaf`) salen del **trial 27** de la optimización bayesiana del notebook
`competencia_01.ipynb` (Optuna/TPE con `n_startup_trials=8` y semilla 285937, 30 trials, métrica AUC en LightGBM, sin undersampling,
entrenando en 202103-202105 y validando en 202106). Sus resultados completos están guardados en `bo_resultados.csv`
(30 trials; el mejor es el trial 27, AUC 0,90468: 1.050 iteraciones, 171 hojas, `min_sum_hessian_in_leaf` 7,287).
Los hiperparámetros fijos son: `max_bin=31`, `min_data_in_leaf=0`, `feature_fraction=0,50`, `feature_fraction_bynode=0,20`, `learning_rate=0,005`.
Correr la optimización desde cero lleva unas 5 horas, por eso aquí se leen los resultados guardados.

In [6]:
bo = pd.read_csv("bo_resultados.csv") if Path("bo_resultados.csv").exists() else pd.read_csv(Path("..") / "exp" / "bo_resultados.csv")
bo = bo.sort_values("value", ascending=False)
mejor = bo.iloc[0]
assert int(mejor["params_num_leaves"]) == PARAMS["num_leaves"] and int(mejor["params_num_iterations"]) == NUM_ITERATIONS
assert abs(float(mejor["params_min_sum_hessian_in_leaf"]) - PARAMS["min_sum_hessian_in_leaf"]) < 1e-9
print("trial", int(mejor["number"]), "| AUC", round(mejor["value"], 5), "| hiperparametros coinciden con los usados abajo")
bo[["number", "value", "params_num_leaves", "params_min_sum_hessian_in_leaf", "params_num_iterations"]].head(8)

trial 27 | AUC 0.90468 | hiperparametros coinciden con los usados abajo


,number,value,params_num_leaves,params_min_sum_hessian_in_leaf,params_num_iterations
0,27,0.904683,171,7.287214,1050
1,26,0.904017,284,25.977212,1357
2,22,0.903706,91,10.095138,1543
3,10,0.903668,128,28.743867,2217
4,12,0.903608,561,26.637233,2054
5,29,0.903479,175,2.513890,843
6,30,0.903383,340,3.662432,767
7,20,0.903382,85,19.058102,1526


## 2.1 Final training
Se entrenan 20 modelos con **hiperparámetros idénticos**, cambiando solo la `seed` de LightGBM, en 202103-202106 y sin ningún tipo de undersampling.
Se aplican a los datos de 202108 (sección 2.2).

**Nota de reproducibilidad.** LightGBM discretiza las variables en *bins* la primera vez que se usa un `Dataset`, y esa construcción depende de la semilla
del primer entrenamiento que lo usa. Las 20 semillas se entrenaron en **dos tandas de 10** (semillas 1 a 10 y 11 a 20), cada una con su propio `Dataset`
construido en el primer uso. Para reproducir exactamente el archivo entregado se replica esa estructura: se crea un `Dataset` nuevo al empezar cada tanda.

In [7]:
fin = ds[ds.foto_mes.isin(MESES_FINAL)]
X_fin = fin[FEATURES].to_numpy(np.float32)
y_fin = fin["y_pos"].to_numpy()

def crear_dataset():
    return lgb.Dataset(X_fin, y_fin, params=DS_PARAMS)
apl = ds[ds.foto_mes == MES_APLICAR]
X_apl = apl[FEATURES].to_numpy(np.float32)
ids_apl = apl["numero_de_cliente"].to_numpy().round().astype("int64")
assert len(set(ids_apl)) == len(ids_apl)
print("entrenamiento:", fin.shape[0], "filas | agosto:", apl.shape[0], "clientes")

preds = {}
for i, sem in enumerate(SEMILLAS):
    t0 = time.time()
    if i in (0, 10):          # un Dataset nuevo al empezar cada tanda de 10 semillas (ver la nota de arriba)
        dfinal = crear_dataset()
    modelo = lgb.train(dict(PARAMS, seed=sem), dfinal, num_boost_round=NUM_ITERATIONS)
    preds[sem] = modelo.predict(X_apl).astype("float32")
    np.save(OUT_DIR / f"pred_202108_seed{sem}.npy", preds[sem])
    print(f"modelo {i + 1}/{len(SEMILLAS)}  semilla {sem}  {time.time() - t0:.0f}s", flush=True)

entrenamiento: 654066 filas | agosto: 164647 clientes


modelo 1/20  semilla 285937  461s


modelo 2/20  semilla 474619  434s


modelo 3/20  semilla 502487  450s


modelo 4/20  semilla 600673  432s


...

modelo 16/20  semilla 362927  420s


modelo 17/20  semilla 130307  426s


modelo 18/20  semilla 212227  422s


modelo 19/20  semilla 301331  421s


modelo 20/20  semilla 915727  420s


## 2.2 Scoring y ensamble
Cada semilla da un ranking de los clientes de 202108. El ensamble promedia las probabilidades por `numero_de_cliente`.

In [8]:
# promedio de las probabilidades de las 20 semillas, por numero_de_cliente (mismo orden de clientes en todas las semillas)
prob = np.mean(np.vstack([preds[s] for s in SEMILLAS]), axis=0)
orden = np.argsort(-prob)
ids_ordenados = ids_apl[orden]                        # clientes de mayor a menor probabilidad
scores = pd.DataFrame({"numero_de_cliente": ids_ordenados, "prob": prob[orden]})
assert scores.numero_de_cliente.is_unique
scores.to_csv(OUT_DIR / "scores_202108.csv", index=False)
print(scores.shape)
scores.head()

(164647, 2)


,numero_de_cliente,prob
0,42425357,0.902690
1,43393631,0.854499
2,44781626,0.841401
3,46863379,0.808191
4,60622229,0.794973


## 2.3 Classification y archivos para el bot
Formato exigido por `@competencia-uno`: `.csv` **sin encabezado**, una sola columna con los `numero_de_cliente` a estimular (enteros), entre 8.000 y 15.000 registros por archivo.
Se generan varios cortes K para compararlos contra el pseudo Kaggle; el entregado es **K = 13.000**.

In [9]:
def guardar_ids(ids, ruta):
    ids = np.asarray(ids).astype("int64")
    assert 8000 <= len(ids) <= 15000 and len(set(ids)) == len(ids)
    pd.DataFrame({"id": ids}).to_csv(ruta, header=False, index=False, lineterminator="\r\n")   # mismos bytes en cualquier sistema operativo

ENT = OUT_DIR / "entregas"
ENT.mkdir(exist_ok=True)
for k in sorted(set(KS + [K_ENTREGA])):
    guardar_ids(ids_ordenados[:k], ENT / f"ensamble_K{k}.csv")

json.dump(dict(semillas=SEMILLAS, ks=KS, k_entrega=K_ENTREGA, params=PARAMS, num_iterations=NUM_ITERATIONS,
               variables_eliminadas=VARIABLES_ELIMINADAS, meses_final=MESES_FINAL, mes_aplicar=MES_APLICAR),
          open(OUT_DIR / "config.json", "w"), indent=2, default=str)
print("archivos en", ENT.resolve())
print("archivo entregado:", ENT / f"ensamble_K{K_ENTREGA}.csv", "|", K_ENTREGA, "clientes")

archivos en salida\entregas
archivo entregado: salida\entregas\ensamble_K13000.csv | 13000 clientes


## Verificación
Compara el archivo generado aquí con el archivo exacto que se subió como entrega final (`ensamble_K13000_ENTREGADO.csv`, ganancia pública 105,0225).

In [10]:
ENTREGADO = Path("ensamble_K13000_ENTREGADO.csv")
if ENTREGADO.exists():
    a = pd.read_csv(ENT / f"ensamble_K{K_ENTREGA}.csv", header=None)[0].to_numpy()
    b = pd.read_csv(ENTREGADO, header=None)[0].to_numpy()
    print("mismos clientes (como conjunto):", set(a) == set(b))
    print("mismo orden fila por fila      :", bool((a == b).all()))
    print("archivos identicos byte a byte :", (ENT / f"ensamble_K{K_ENTREGA}.csv").read_bytes() == ENTREGADO.read_bytes())
else:
    print("no se encontro ensamble_K13000_ENTREGADO.csv en esta carpeta; nada que comparar")

mismos clientes (como conjunto): True
mismo orden fila por fila      : True
archivos identicos byte a byte : True
